# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook explores the FAIR² protein dataset using the `mlcroissant` library, following the [Croissant schema](https://mlcommons.org/croissant/spec/).

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded!")
print("Title:", metadata.name)
print("Description:", metadata.description)
print(f"Version: {getattr(metadata, 'version', None)}" )

# Print citation
print("Citation:", getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', None)))

## 2. Data Overview
Explore available record sets, fields, and their `@id` values from the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets in the metadata.\n")
overview = []
for rs in record_sets:
    print(f"RecordSet: @id={rs.id}, name={rs.name}")
    print("  Fields:")
    for f in rs.fields:
        details = [f"@id={f.id}", f"name={f.name}", f"dataType={getattr(f, 'data_type', None)}"]
        if hasattr(f, 'column') and f.column is not None:
            details.append(f"column: @id={f.column.id if hasattr(f.column, 'id') else f.column}")
        print("    - ", ", ".join(details))
        overview.append({'record_set_id': rs.id, 'field_id': f.id, 'field_name': f.name, 'data_type': getattr(f, 'data_type', None)})

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. Use the record set and field `@id` values from the previous overview.

In [ ]:
# Extract all record set @ids for loading
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"RecordSet IDs: {record_set_ids}\n")

dataframes = {}
for rs_id in record_set_ids:
    # Load all records for this record set
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  DataFrame: {df.shape[0]} rows, {df.shape[1]} columns.")
        print(f"  Columns: {df.columns.tolist()}")
        print()
    else:
        print("  No records found for this record set.\n")

# For demonstration, select the first non-empty record set for downstream analysis
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        main_record_set_id = rs_id
        break
if main_record_set_id:
    print(f"Selected RecordSet for analysis: {main_record_set_id}\n")
    print(dataframes[main_record_set_id].head())
else:
    print("No record set with data was found.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps, such as filtering records by numeric field, normalization, and grouping.

In [ ]:
import numpy as np
# Identify a numeric field
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to automatically detect a numeric column
    numeric_field_id = None
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field_id = col
                break
        except Exception:
            continue
    if numeric_field_id is not None:
        print(f"Using numeric field for EDA: {numeric_field_id}")
        # Example: Filter values above median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > median ({threshold}):")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a categorical field for grouping (the first non-numeric column)
        group_field = None
        for col in df.columns:
            if not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break
        if group_field is not None:
            print(f"Grouping by: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No suitable numeric field found for analysis.")
else:
    print("No main record set DataFrame available.")

## 5. Visualization
Visualize the distribution of a numeric field and its grouped means by a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Visualize filtered numeric field
if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, visualize group means
    if group_field is not None:
        group_means = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 4))
        sns.barplot(data=group_means, x=group_field, y=numeric_field_id, palette='viridis')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and performed initial data processing and visualization on the FAIR² dataset using the `mlcroissant` library. For advanced analysis, refer to the Croissant metadata to select relevant entities using their `@id` references.

- We identified the available record sets and fields by their unique `@id`s.
- We extracted data from a main record set, performed filtering and normalization on a numeric field, and grouped data by a categorical attribute.
- Finally, we visualized the numeric data distribution and category-wise means.

This demonstrates a scalable, metadata-guided approach for reproducible data exploration with FAIR datasets.